## Investigating model behavior with markdown token prompts

In [4]:
import os
import torch
import transformers
import jlens

assert torch.cuda.is_available(), "No CUDA GPU is visible to this notebook"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
print(f"PyTorch: {torch.__version__}; Transformers: {transformers.__version__}")
print("Hugging Face cache:", os.environ.get("HF_HOME", "~/.cache/huggingface"))

GPU: NVIDIA L40S (44.4 GiB)
PyTorch: 2.9.1+cu128; Transformers: 5.16.1
Hugging Face cache: /root/.cache/huggingface


## Load Qwen

Qwen3.5-9B is packaged as a multimodal model. `jlens` automatically selects its text decoder, which is the activation basis Camila's checkpoints target.

In [5]:
from transformers import AutoModelForMultimodalLM, AutoTokenizer

# Compatibility-locked to Camila Blank's qwen3.5-9b lens checkpoints.
# Do not substitute a base/fine-tuned/quantized model or add adapters.
MODEL_ID = "Qwen/Qwen3.5-9B"
LENS_REPO = "camilablank/workspace-lenses"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
hf_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
hf_model.eval()
print("Loaded:", type(hf_model).__name__)
print(f"CUDA allocated: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Loaded: Qwen3_5ForConditionalGeneration
CUDA allocated: 35.1 GiB


In [6]:
lens_model = jlens.from_hf(hf_model, tokenizer)
j_lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename="qwen3.5-9b/j-lens/lens.pt"
)
r_lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename="qwen3.5-9b/r-lens/lens.pt"
)

assert lens_model.d_model == j_lens.d_model == r_lens.d_model == 4096
print(lens_model)
print("J-lens:", j_lens)
print("R-lens:", r_lens)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

HFLensModel(Qwen3_5ForConditionalGeneration, n_layers=32, d_model=4096)
J-lens: JacobianLens(d_model=4096, n_prompts=25, source_layers=[0..30] (31 layers))
R-lens: JacobianLens(d_model=4096, n_prompts=25, source_layers=[0..30] (31 layers))


## Raw text completion

This intentionally tokenizes `PROMPT` directly: there is no API, system message, chat template, assistant prefix, or hidden intermediate input. The exposed controls affect only the prompt or decoding; they do **not** modify the model or lens weights.

Compatibility-sensitive choices remain fixed above: exact model ID, tokenizer, BF16 weights, no quantization/adapters, and Camila Blank's matching J/R checkpoints. Changing any of those requires a matching lens or a new validation. The full generated completion—including special/reasoning tokens—is streamed below and retained in `completion_text`.

In [21]:
import torch
from transformers import LogitsProcessor, LogitsProcessorList, TextStreamer

MODE = "elicit"
# Options:
# "thinking_general"
# "thinking_coding"
# "instruct_general"
# "instruct_reasoning"

QUESTION = "----------"
MAX_NEW_TOKENS = 384
SEED = 2029

PRESETS = {
    "thinking_general": {
        "enable_thinking": True,
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
    },
    "thinking_coding": {
        "enable_thinking": True,
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 0.0,
        "repetition_penalty": 1.0,
    },
    "instruct_general": {
        "enable_thinking": False,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
    },
    "instruct_reasoning": {
        "enable_thinking": False,
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
    },
    "elicit": {
        "enable_thinking": True,
        "temperature": 1.0,
        "top_p": 1,
        "top_k": 20,
        "min_p": 0.0,
        "presence_penalty": 0,
        "repetition_penalty": 1.0,
    },
}


In [22]:

class PresencePenalty(LogitsProcessor):
    """Subtract a fixed penalty from every token already generated."""

    def __init__(self, penalty: float, prompt_length: int):
        self.penalty = penalty
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated = input_ids[:, self.prompt_length :]
        if self.penalty and generated.shape[1]:
            for batch_index in range(generated.shape[0]):
                seen = generated[batch_index].unique()
                scores[batch_index, seen] -= self.penalty
        return scores

cfg = PRESETS[MODE]

messages = [{"role": "user", "content": QUESTION}]
serialized_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=cfg["enable_thinking"],
)

print("Exact serialized prompt:", repr(serialized_prompt))


model_inputs = tokenizer(
    serialized_prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(hf_model.device)

prompt_length = model_inputs["input_ids"].shape[-1]
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

with torch.inference_mode():
    generated_ids = hf_model.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        top_k=cfg["top_k"],
        min_p=cfg["min_p"],
        repetition_penalty=cfg["repetition_penalty"],
        logits_processor=LogitsProcessorList([
            PresencePenalty(cfg["presence_penalty"], prompt_length)
        ]),
        use_cache=True,
        streamer=streamer,
    )

completion_ids = generated_ids[0, prompt_length:]
completion_text = tokenizer.decode(
    completion_ids,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

print(f"\n--- {len(completion_ids)} generated tokens ---")

Exact serialized prompt: '<|im_start|>user\n----------<|im_end|>\n<|im_start|>assistant\n<think>\n'
Okay, the user just sent a message with just a separator line. Hmm, that's pretty minimal. Maybe they're testing the system? Or maybe they made a mistake and hit enter too quickly. Let me check the rules.

The conversation history has 7 entries, which is more than the 20 token limit. Wait, no, the system says to use the most recent 20k tokens when deciding context. But here, the history is 7 steps, so I should consider all prior messages for context.

Current user's message is just "----------", which could be a mistake. They might want to continue the conversation but accidentally sent a separator. Or maybe they're trying to indicate a break? But according to my guidelines, I should avoid assumptions. Maybe they need help, or they’re confused. I should check previous interactions.

Looking at the history, the last user message was "11:59:22 2024-04-12" followed by an assistant response 

 Generation is matched against open API responses. Params are equivalent.

In [ ]:
PROMPT = "==="
LAYERS = [layer for layer in [4, 8, 12, 16, 20, 24, 28, 30] if layer in j_lens.source_layers]

def top_tokens(logits, k=5):
    token_ids = logits[0].topk(k).indices.tolist()
    return [repr(tokenizer.decode([token_id])) for token_id in token_ids]

def run_lens(name, lens):
    lens_logits, model_logits, input_ids = lens.apply(
        lens_model, PROMPT, layers=LAYERS, positions=[-1]
    )
    print(f"\n{name}: {PROMPT!r}")
    for layer in LAYERS:
        print(f"layer {layer:>2}:", top_tokens(lens_logits[layer]))
    print("model   :", top_tokens(model_logits))
    return lens_logits

j_logits = run_lens("J-lens", j_lens)
r_logits = run_lens("R-lens", r_lens)